# Falcon-H1-3B Layer Experiments — Google Colab
Run specific layers of the patch-effect experiment.
Set `START_LAYER` and `END_LAYER` below, then **Runtime → Run all**.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
HF_TOKEN     = ""          # paste your HuggingFace token
START_LAYER  = 0           # first layer to run (inclusive)
END_LAYER    = 7           # last  layer to run (inclusive)
NUM_INSTANCES = 100
NUM_SAMPLES   = 1000
MODEL_ID      = "tiiuae/Falcon-H1-3B-Instruct"
BRANCH        = "fix/multi-token-box-prediction"
REPO_URL      = "https://github.com/NitzanZacharia/100rep.git"
OUTPUT_DIR    = "/content/results"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU found — change Runtime type to GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo
import os
if not os.path.isdir("/content/100rep"):
    !git clone -b {BRANCH} {REPO_URL} /content/100rep
else:
    !git -C /content/100rep pull
%cd /content/100rep

In [ ]:
# Install dependencies
!pip install -q transformers accelerate datasets tqdm matplotlib pandas

# CausalAbstraction sub-package expected at this path by the script
import sys
if "/content/100rep" not in sys.path:
    sys.path.insert(0, "/content/100rep")
if os.path.isdir("/content/100rep/CausalAbstraction"):
    sys.path.append("/content/100rep/CausalAbstraction")

In [ ]:
# Set memory allocator (Colab PyTorch is recent enough for expandable_segments)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"

!mkdir -p {OUTPUT_DIR} /content/100rep/logs

In [ ]:
# Run experiment
cmd = (
    f"python run_layer_experiments.py "
    f"--model-id '{MODEL_ID}' "
    f"--output-dir '{OUTPUT_DIR}' "
    f"--num-instances {NUM_INSTANCES} "
    f"--num-samples {NUM_SAMPLES} "
    f"--start-layer {START_LAYER} "
    f"--end-layer {END_LAYER} "
    f"--trust-remote-code "
    + (f"--hf-token '{HF_TOKEN}' " if HF_TOKEN else "")
)
print("Running:", cmd)
!{cmd}

In [ ]:
# Download results as a zip
import shutil
zip_path = "/content/results.zip"
shutil.make_archive("/content/results", "zip", OUTPUT_DIR)

from google.colab import files
files.download(zip_path)